# Dogs vs Cats: transfer learning report

This notebook reads report tables from `runs/report/summary/`. Local image previews are optional and are skipped when `data/` is absent.

In [ ]:
from pathlib import Path

repo = Path.cwd()
if not (repo / 'runs/report/summary').is_dir(): repo = repo.parent
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

summary = repo / 'runs/report/summary'
matrix = pd.read_csv(summary / 'matrix.csv')
curves = pd.read_csv(summary / 'curves.csv')
seeds = pd.read_csv(summary / 'seeds.csv')
confusion = pd.read_csv(summary / 'confusion.csv')
misclassified = pd.read_csv(summary / 'misclassified.csv')


## a. Data

The training split contains the cleaned cat and dog images, and validation is balanced closely across the two labels. Six exact duplicates were removed during the handoff cleanup. The sample grid is shown only when the local dataset is available.

In [ ]:
counts = json.loads((summary / 'data_counts.json').read_text())['counts']
labels = ['train/cat', 'train/dog', 'val/cat', 'val/dog', 'test']
fig, ax = plt.subplots(figsize=(8, 3)); ax.bar(labels, [counts[k] for k in labels]); ax.set_ylabel('Images'); ax.tick_params(axis='x', rotation=25); plt.show()
from PIL import Image, ImageEnhance
from IPython.display import display
if Path('data').is_dir():
    samples = [next(Path('data', split).glob('*')) for split in ('train/cat','train/dog','val/cat','val/dog') if Path('data', split).is_dir() and any(Path('data', split).glob('*'))]
    if samples:
        fig, axes = plt.subplots(1, len(samples), figsize=(3*len(samples), 3)); axes = np.atleast_1d(axes)
        for ax, path in zip(axes, samples): ax.imshow(Image.open(path)); ax.set_title(path.parent.name); ax.axis('off')
        plt.show()
else: print('Local data/ is absent; image grid skipped.')

## b. Pipelines

Baseline uses the original training files; augmented replaces each training file with one seeded adjustment; mixed retains originals and adds one adjusted copy. The examples below show source images for the four adjustment types, so the preview does not require adjusted copies in the notebook.

In [ ]:
adjustments = pd.read_csv(summary / 'adjustments.csv')
print(adjustments.to_string(index=False))
examples = pd.read_csv(summary / 'adjustment_examples.csv')
def apply_example(image, adjustment, parameter):
    if adjustment == 'horizontal_flip': return image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
    values = dict(item.split('=', 1) for item in parameter.split(';') if '=' in item)
    if adjustment == 'rotation': return image.rotate(float(values['angle']), resample=Image.Resampling.BILINEAR)
    if adjustment == 'random_resized_crop':
        left, top, width, height = (int(values[key]) for key in ('left','top','width','height'))
        return image.crop((left,top,left+width,top+height)).resize((224,224),Image.Resampling.BILINEAR)
    if adjustment == 'colour_jitter':
        for enhancer,key in zip((ImageEnhance.Brightness,ImageEnhance.Contrast,ImageEnhance.Color),('brightness','contrast','saturation')):
            image=enhancer(image).enhance(float(values[key]))
        return image
    raise ValueError(adjustment)
if (repo / 'data').is_dir():
    fig, axes = plt.subplots(1, len(examples), figsize=(4*len(examples), 3)); axes = np.atleast_1d(axes)
    for ax, row in zip(axes, examples.itertuples()):
        path = repo / 'data' / row.source_path
        if path.is_file(): ax.imshow(apply_example(Image.open(path).convert('RGB'),row.adjustment,row.parameter)); ax.set_title(row.adjustment); ax.axis('off')
        else: ax.set_visible(False)
    plt.show()
else: print('Local data/ is absent; adjustment image examples skipped.')

## c. Main result

Each bar compares one completed seed-42 run, so these cells describe observed validation accuracy rather than an average over seeds. Frozen means head only, last_block means last block plus head, and full means the full network is trainable. The axis is zoomed to the measured range; the table retains all 18 individual values.

In [ ]:
configs = matrix.backbone + ' / ' + matrix.freeze
wide = matrix.pivot(index=['backbone','freeze'], columns='pipeline', values='val_acc').reindex(pd.MultiIndex.from_product([['resnet18','resnet50'],['frozen','last_block','full']]))
ax = wide.plot(kind='bar', figsize=(12,4), ylim=(max(.0, matrix.val_acc.min()-.02), min(1, matrix.val_acc.max()+.005)))
ax.set_ylabel('Validation accuracy'); ax.set_xlabel('Backbone / freeze mode'); ax.legend(title='Pipeline'); plt.tight_layout(); plt.show()
display(matrix[['backbone','freeze','pipeline','val_acc','val_loss','epochs','sec_per_epoch']].round(4))

## d. Per-backbone view

These lines show how validation accuracy changes as more of each network is trainable. Pipeline-to-pipeline changes are visible within each freeze mode, while the single seed per matrix cell limits claims about stability.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10,4), sharey=True)
for ax, backbone in zip(axes, ['resnet18','resnet50']):
    subset=matrix[matrix.backbone==backbone]
    for pipeline, group in subset.groupby('pipeline'):
        group=group.set_index('freeze').reindex(['frozen','last_block','full']); ax.plot(group.index, group.val_acc, marker='o', label=pipeline)
    ax.set_title(backbone); ax.set_xlabel('Freeze mode'); ax.grid(alpha=.25)
axes[0].set_ylabel('Validation accuracy'); axes[-1].legend(); plt.tight_layout(); plt.show()

## e. Cost

Training time is estimated from the mean epoch duration multiplied by the recorded epoch count. Frozen feature extraction is far cheaper in these runs, but the ratio reflects this machine, cached features and these exact runs.

In [ ]:
cost=matrix.copy(); cost['minutes']=cost.sec_per_epoch*cost.epochs/60; cost['label']=cost.backbone.str.replace('resnet','R')+' '+cost.freeze+' '+cost.pipeline
fig, ax=plt.subplots(figsize=(10,5))
for _,r in cost.iterrows(): ax.scatter(r.minutes,r.val_acc); ax.annotate(r.label,(r.minutes,r.val_acc),fontsize=7,xytext=(3,3),textcoords='offset points')
ax.set_xlabel('Estimated training minutes'); ax.set_ylabel('Validation accuracy'); ax.grid(alpha=.25); plt.show()
print('Median non-frozen / frozen time ratio:', round(cost[cost.freeze!='frozen'].minutes.median()/cost[cost.freeze=='frozen'].minutes.median(),1))

## f. Training curves

Curves use each epoch logged before early stopping, with the validation-accuracy-selected epoch marked in every panel. The charts show optimization behavior for these runs; they do not replace the final validation comparison.

In [ ]:
fig, axes=plt.subplots(2,3,figsize=(13,7),sharex=True)
for ax,(backbone,freeze) in zip(axes.flat,[(b,f) for b in ['resnet18','resnet50'] for f in ['frozen','last_block','full']]):
    for pipeline,g in curves[(curves.backbone==backbone)&(curves.freeze==freeze)].groupby('pipeline'):
        g=g.sort_values('epoch'); ax.plot(g.epoch,g.train_loss,ls='--',alpha=.65); ax.plot(g.epoch,g.val_loss,label=pipeline)
        stops=g[g.early_stop.astype(bool)]
        if not stops.empty: ax.axvline(stops.epoch.max(),color='black',ls=':',alpha=.55)
    ax.set_title(f'{backbone} / {freeze}'); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
axes[0,0].legend(fontsize=7); plt.tight_layout(); plt.show()
print('Solid lines are validation loss; dashed lines are training loss. Dotted marks indicate runs that stopped before the configured epoch limit.')

## g. Seed noise

The three-seed groups estimate run-to-run spread for the frozen ResNet-50 setting. Matrix differences smaller than the pooled seed standard deviation are not clearly larger than the observed seed variation; all other matrix cells still have only one seed.

In [ ]:
seed_summary=seeds.groupby('pipeline').val_acc.agg(['mean','std']).reindex(['best','aug','mixed']); fig,ax=plt.subplots(figsize=(7,4))
for x,(pipeline,row) in enumerate(seed_summary.iterrows()):
    vals=seeds[seeds.pipeline==pipeline].val_acc; ax.errorbar(x,row['mean'],yerr=row['std'],fmt='o',capsize=5); ax.scatter([x]*len(vals),vals,alpha=.65)
ax.set_xticks(range(3),seed_summary.index); ax.set_ylabel('Validation accuracy'); ax.set_title('ResNet-50 frozen: mean ? sample SD'); ax.grid(axis='y',alpha=.25); plt.show()
spread=seed_summary['std'].mean(); print(f'Mean within-pipeline seed SD: {spread:.5f}. Matrix differences below this are smaller than the typical observed seed spread.')

## h. Errors

The confusion matrix and error list use the highest-accuracy matrix run, selected from validation accuracy. Images are shown only when the local data files can be resolved; the numeric error table remains available without photos.

In [ ]:
best=matrix.loc[matrix.val_acc.idxmax()]; cm=confusion[(confusion.backbone==best.backbone)&(confusion.freeze==best.freeze)&(confusion.pipeline==best.pipeline)].iloc[0]
cm_values=np.array([[cm.tn,cm.fp],[cm.fn,cm.tp]]); fig,ax=plt.subplots(); ax.imshow(cm_values,cmap='Blues'); ax.set_xticks([0,1],['cat','dog']); ax.set_yticks([0,1],['cat','dog']); ax.set_xlabel('Predicted'); ax.set_ylabel('True')
for (i,j),v in np.ndenumerate(cm_values): ax.text(j,i,str(v),ha='center',va='center')
ax.set_title(f'{best.backbone} / {best.freeze} / {best.pipeline}'); plt.show()
display(misclassified.head(30))
if Path('data').is_dir():
    shown=0
    for row in misclassified.head(12).itertuples():
        path=Path(row.path)
        if not path.is_absolute(): path=Path.cwd()/path
        if path.is_file():
            display(Image.open(path).resize((160,120))); print(f'true={row.true}, predicted={row.pred}, p(dog)={float(row.prob):.3f}'); shown+=1
    print(f'Showing {shown} available error images.')
else: print('Local data/ is absent; error image grid skipped.')

## i. Submission

The label split counts come from the saved submission file, and confidence is summarized from its paired prediction probabilities. This describes output balance and certainty, not test accuracy because test labels are unavailable.

In [ ]:
stats=json.loads((summary/'submission_stats.json').read_text()); print('Submission labels:',stats['labels'])
hist=pd.DataFrame(stats['confidence_histogram']); fig,ax=plt.subplots(figsize=(8,3)); ax.bar(hist['range'],hist['count']); ax.set_xlabel('Maximum class probability'); ax.set_ylabel('Test images'); ax.tick_params(axis='x',rotation=30); plt.show()

## What we conclude

The table supports direct comparisons among these completed runs, and the seed groups show how much the frozen ResNet-50 results varied across the three recorded seeds. Any differences in other configurations are descriptive because they have one seed each; the test submission histogram measures confidence, not correctness.